# EKF G/F · Fracción del YAN del pulso destinada a biomasa

Se compara la variante G/F del EKF 16 con un reparto del YAN añadido según el azúcar residual. Se conservan los parámetros cinéticos, los rendimientos, los parámetros de gas, las covarianzas y las entradas del 16. LAB016–021 usan sus ventanas observadas. LAB022–024 se corren con el CSV interpolado y con el hueco original de las 22 h. Los datos de laboratorio posteriores a los iniciales se usan para evaluar.

En cada pulso se fija \(\alpha=\mathrm{clip}((G+F)_{t^-}/S_0,0,1)\) con el estado justo antes de añadir el YAN. El YAN previo y el recién añadido se mezclan en proporción a su concentración. Como todos los aportes se captan proporcionalmente, basta mantener la fracción efectiva \(f_X\) del YAN presente destinada a biomasa. Entre pulsos se conserva esa fracción y las correcciones de YAN del EKF se reparten en la misma proporción.

Se mantiene la ley de captación total del 16. Solo \(f_X\) de esa captación produce biomasa, consume azúcar por crecimiento y aporta CO₂ por crecimiento. El YAN restante se consume por una vía sin efecto fermentativo directo en esta prueba. La fermentación de G y F sigue dependiendo del estado estimado, incluida la biomasa. La propagación de covarianza usa las sensibilidades de la nueva dinámica condicionadas a la fracción calculada al pulso.

## Configuración y entradas de LAB016–021

In [ ]:
from pathlib import Path
import ast
import json
import os
import sys
from types import ModuleType

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/wine_fermentation").is_dir())
EKF = ROOT / "notebooks/EKF"
HERE = EKF / "kalman_analysis"
prefix = Path(os.environ["CONDA_PREFIX"]).resolve()
assert prefix.name.lower() == "fermentation" and Path(sys.prefix).resolve() == prefix
print(f"Kernel {prefix.name} · SciPy {scipy.__version__}")

# Load the existing helpers as Python functions. The filter below uses the
# vectorized NumPy transition from EKF05 and does not require Numba.
core_path = HERE / "analysis_core.py"
tree = ast.parse(core_path.read_text(encoding="utf-8"))
tree.body = [node for node in tree.body
             if not (isinstance(node, ast.ImportFrom) and node.module == "numba")]
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        node.decorator_list = [
            d for d in node.decorator_list
            if not (isinstance(d, ast.Call) and isinstance(d.func, ast.Name) and d.func.id == "njit")
        ]
core = ModuleType("analysis_core")
core.__file__ = str(core_path)
exec(compile(tree, str(core_path), "exec"), vars(core))

# Reuse the six-lot input preparation from the four-source comparison.
reference = HERE / "04_gf_variable_solubility_fourth_noise_source.source.ipynb"
reference_nb = nbformat.read(reference, as_version=4)
preparation = next(cell.source for cell in reference_nb.cells
                   if cell.cell_type == "code" and "DATA,CHEM,AUDIT,_=core.prepare(ROOT)" in cell.source)
assert preparation.count("import analysis_core as core") == 1
preparation = preparation.replace("import analysis_core as core", "")
exec(compile(preparation.replace("display(AUDIT)", ""), str(reference), "exec"))

calibration = ROOT / "notebooks/diagnostics/zenteno_lab004_009_incertidumbre_exploratoria.ipynb"
saved_nb = nbformat.read(calibration, as_version=4)
saved = json.loads(saved_nb.cells[-1].outputs[-1].text)
rate_names = ["mu0", "betaG0", "betaF0"]
assert saved["orden_D_theta"] == rate_names
D_theta = np.array(saved["D_theta_Yef_fijo"], dtype=float)
np.testing.assert_array_equal(D_theta, np.diag(np.diag(D_theta)))
assert (np.diag(D_theta) > 0).all()
R = 0.00218
R_cell = next(cell.source for cell in reference_nb.cells
              if cell.cell_type == "code" and "R = 0.00218" in cell.source)
assert ast.literal_eval(ast.parse(R_cell).body[0].value) == R
theta = core.THETA
nominal = np.array([theta[name] for name in rate_names])
sigma_theta = np.sqrt(np.diag(D_theta))
SCALE = core.SCALE
scale_outer = np.outer(SCALE, SCALE)
KLA, GAIN = core.ARCHIVED_GAS
PCO2_BAR = 1.0


## Muestras y señales de LAB022–024

La lectura y selección de muestras sigue la preparación de datos de LAB022–024. Para comparar biomasa, la concentración viable Oculyze en millones de células/mL se multiplica por el factor histórico de 0,03 g/L por millón de células/mL.

In [ ]:
import csv
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd

%matplotlib inline

cwd = Path.cwd().resolve()
data_dir = next(
    path / 'data/original/natural/lab022_024'
    for path in (cwd, *cwd.parents)
    if (path / 'data/original/natural/lab022_024').is_dir()
)
labs = ['LAB022', 'LAB023', 'LAB024']
workbook = data_dir / 'lab022-024MedicionesOffline_estandar.xlsx'
experiments = pd.read_excel(workbook, sheet_name='Experimentos').set_index('CodigoLab').loc[labs]
inoculation = experiments['FechaHora_Inoculacion']

offline = (
    pd.read_excel(workbook, sheet_name='MedicionesOffline')
    .rename(columns={'CodigoLab': 'lab', 'Muestra': 'sample'})
    .loc[lambda frame: frame['lab'].isin(labs)]
    .copy()
)
offline['sampled_at'] = [
    datetime.combine(date.date(), time)
    for date, time in zip(offline['Fecha'], offline['Hora'])
]
nutrition = (
    offline.loc[offline['Observaciones'].eq('Hora de nutrición')]
    .set_index('lab')['sampled_at']
    .loc[labs]
)
offline = offline.dropna(subset='sample').copy()
offline['sample'] = offline['sample'].replace(
    {**{number: f'{number:02d}' for number in range(1, 7)}, 'POST NUT': 'POSTNUT'}
)
offline = offline.sort_values(['lab', 'sampled_at'])

def elapsed_hours(frame):
    return (frame['sampled_at'] - frame['lab'].map(inoculation)).dt.total_seconds() / 3600

offline['hours'] = elapsed_hours(offline)
sample_times = offline[['lab', 'sample', 'sampled_at']]
post_times = sample_times.loc[sample_times['sample'].eq('POSTNUT')].set_index('lab')['sampled_at']
assert post_times.eq(pd.Timestamp('2026-11-09 11:30')).all()
assert post_times.gt(nutrition).all()


y15_sources = {
    'EXPAuto(2026-11-02 11-57).txt': {'3': '01', '02': '02', '03': '03', '04': '04'},
    'ONLINE(26-11-09 09-52)_1.txt': {'05': '05', '06': '06', 'POSTNUT': 'POSTNUT'},
}
y15_columns = ['source_label', 'assay', 'station', 'reading', 'unit', 'assayed_at']
assays = ['PAN', 'AMMONIA', 'YAN', 'GLUCOSE-320', 'GLU-FRU-320']
y15_frames = []
for filename, samples in y15_sources.items():
    labels = {
        f'{lab}-{suffix}': (lab, sample)
        for lab in labs
        for suffix, sample in samples.items()
    }
    raw = pd.read_csv(
        data_dir / filename, sep='\t', header=None, names=y15_columns,
        dtype={'reading': str}, encoding='utf-8-sig',
    )
    keep = raw['source_label'].isin(labels) & raw['assay'].isin(assays)
    selected = raw.loc[keep].drop_duplicates().copy()
    selected[['lab', 'sample']] = selected['source_label'].map(labels).tolist()
    selected['assayed_at'] = pd.to_datetime(selected['assayed_at'], format='%d-%m-%Y %H:%M:%S')
    selected['reading'] = pd.to_numeric(selected['reading'].str.replace(',', '.', regex=False))
    selected['source_file'] = filename
    y15_frames.append(selected)

y15 = pd.concat(y15_frames, ignore_index=True).merge(
    sample_times, on=['lab', 'sample'], how='left', validate='many_to_one'
)
assert y15['sampled_at'].notna().all()
assert y15.loc[y15['sample'].eq('01'), 'sampled_at'].eq(
    y15.loc[y15['sample'].eq('01'), 'lab'].map(inoculation)
).all()
assert y15.loc[y15['sample'].eq('POSTNUT'), 'assayed_at'].dt.date.eq(
    datetime(2026, 11, 9).date()
).all()

# Pivot also checks that each lab, sample, and assay has one value.
chemistry = (
    y15.pivot(index=['lab', 'sample', 'sampled_at'], columns='assay', values='reading')
    .reset_index()
    .sort_values(['lab', 'sampled_at'])
)
chemistry['hours'] = elapsed_hours(chemistry)
chemistry['Fructose (derived)'] = chemistry['GLU-FRU-320'] - chemistry['GLUCOSE-320']

# Prefer instrument-reported YAN where available and calculate the remaining values.
ammonia_n_fraction = 0.8225
chemistry['YAN (calculated)'] = chemistry['PAN'] + ammonia_n_fraction * chemistry['AMMONIA']
chemistry['YAN for plot'] = chemistry['YAN'].fillna(chemistry['YAN (calculated)'])

expected_samples = {'01', '02', '03', '04', '05', '06', 'POSTNUT'}
assert chemistry.groupby('lab')['sample'].agg(set).eq(expected_samples).all()
assert chemistry['YAN for plot'].notna().all()
assert chemistry.loc[chemistry['sample'].eq('POSTNUT'), 'YAN'].notna().all()


oculyze_frames = []
for lab in labs:
    filename = data_dir / f'oculyze_{lab.lower()}.csv'
    # Each CSV record is wrapped in a single quoted field.
    with filename.open(encoding='utf-8-sig', newline='') as file:
        records = [row[0] for row in csv.reader(file) if row]
    measurements = pd.DataFrame(csv.DictReader(records))
    labels = measurements['Name']
    if lab == 'LAB023':
        labels = labels.replace({'LAB024-05': 'LAB023-05'})
    measurements[['lab', 'sample']] = labels.str.rsplit('-', n=1, expand=True)
    assert measurements['lab'].eq(lab).all()
    measurements['sampled_at'] = pd.to_datetime(
        measurements['Description'].str.strip(), format='%d/%m/%y %H:%M'
    )
    for column in ['Concentration', 'Viability', 'Density']:
        measurements[column] = pd.to_numeric(measurements[column])
    oculyze_frames.append(measurements)

oculyze = pd.concat(oculyze_frames, ignore_index=True).sort_values(['lab', 'sampled_at'])
assert oculyze.groupby('lab')['sample'].agg(set).eq(expected_samples - {'POSTNUT'}).all()
density_check = oculyze.merge(
    offline[['lab', 'sample', 'Densidad_g_L']], on=['lab', 'sample'], validate='one_to_one'
)
numbered_later = density_check.loc[density_check['sample'].ne('01')]
assert numbered_later['Density'].eq(numbered_later['Densidad_g_L']).all()
oculyze['hours'] = elapsed_hours(oculyze)
oculyze['Viable concentration'] = oculyze['Concentration'] * oculyze['Viability'] / 100

In [ ]:
instrument = json.loads((ROOT / "configs/co2_instrument.json").read_text(encoding="utf-8"))
conversion = 44.0095 * 60 * instrument["co2_correction_factor"] / (1000 * 24.16)
imputed = pd.read_csv(ROOT / "data/processed/lab022_024_co2_30min_imputed.csv",
                      parse_dates=["window_end_local"]).set_index("lab", verify_integrity=True)
assert set(imputed.index) == set(labs)
assert imputed["source"].eq("linear_interpolation_for_plot").all()

reactor_to_lab = pd.Series(experiments.index, index=experiments["Fermentador"])
def backup(filename, column):
    frame = pd.read_csv(data_dir / filename, usecols=["timestamp", "fermentador", column],
                        parse_dates=["timestamp"]).rename(columns={"timestamp": "sampled_at"})
    frame["lab"] = frame["fermentador"].map(reactor_to_lab)
    frame["hours"] = elapsed_hours(frame)
    lower = -1 / 60 if column == "T" else 0.
    frame = frame.loc[frame["hours"].ge(lower)].sort_values(["lab", "sampled_at"])
    assert frame["lab"].notna().all() and not frame.duplicated(["lab", "sampled_at"]).any()
    assert np.isfinite(frame[column]).all()
    return frame

gas = backup("backup_global_co2.csv", "flow_sccm")
temperature = backup("backup_global_temperatura.csv", "T")
nutrition_h = (nutrition - inoculation).dt.total_seconds() / 3600
sfx_mass_g, fda_mass_g = 1.0, 0.4
sfx_yan_equivalent_g_per_g = .10
fda_n_g_per_g = 2 * 14.007 / 132.1
pulse_n_g = sfx_mass_g * sfx_yan_equivalent_g_per_g + fda_mass_g * fda_n_g_per_g
biomass = oculyze.assign(X_g_L=oculyze["Viable concentration"] * .03)
X0 = biomass.loc[biomass["sample"].eq("01")].set_index("lab")
chem0 = chemistry.loc[chemistry["sample"].eq("01")].set_index("lab")
assert X0["hours"].eq(.25).all() and chem0["hours"].eq(0.).all()
assert offline["Vol_Residual_mL"].isna().all()
withdrawals = offline.loc[offline["Observaciones"].eq("se quitan 50ml")]
volumes_initial = experiments["Volumen_Inicial_L"].str.removesuffix("L").astype(float)
assert withdrawals.groupby("lab").size().eq(4).all()

def prepare_lab(lab):
    filled = imputed.loc[lab]
    assert filled["fermentador"] == experiments.at[lab, "Fermentador"]
    fill_h = float(filled["window_end_h"])
    assert filled["window_end_local"] == inoculation[lab] + pd.Timedelta(hours=fill_h)
    removed_at = withdrawals.loc[withdrawals["lab"].eq(lab), "hours"].to_numpy()
    def volume(hours):
        return volumes_initial[lab] - .05 * np.searchsorted(removed_at, hours, side="right")

    raw = gas.loc[gas["lab"].eq(lab) & gas["hours"].gt(0)].copy()
    raw["bin"] = np.ceil(raw["hours"] / .5 - 1e-10).astype(int)
    raw["flow_mass"] = raw["flow_sccm"] * conversion / volume(raw["hours"])
    last_hour = np.floor(raw["hours"].max() / .5) * .5
    rows = []
    for index in range(1, int(last_hour / .5) + 1):
        end = index * .5
        window = raw.loc[raw["bin"].eq(index)]
        max_gap = np.diff(np.r_[end - .5, window["hours"], end]).max()
        valid = len(window) >= 30 and max_gap <= 2 / 60 + 1e-10
        rows.append((end, window["flow_sccm"].mean() if valid else np.nan,
                     window["flow_mass"].mean() if valid else np.nan, valid))
    observed = pd.DataFrame(rows, columns=["hours", "sccm", "measured", "valid"]).set_index("hours")
    assert observed.index[~observed["valid"]].tolist() == [fill_h]
    endpoints = [filled["left_window_end_h"], filled["right_window_end_h"]]
    endpoint_flow = [filled["left_mean_sccm"], filled["right_mean_sccm"]]
    np.testing.assert_allclose(observed.loc[endpoints, "sccm"], endpoint_flow, rtol=0, atol=1e-8)
    np.testing.assert_allclose(filled["flow_mean_sccm"], np.interp(fill_h, endpoints, endpoint_flow),
                               rtol=0, atol=1e-8)
    assert volume(fill_h - .5) == volume(fill_h)
    augmented = observed.copy()
    augmented.loc[fill_h, ["sccm", "measured", "valid"]] = [
        filled["flow_mean_sccm"], filled["flow_mean_sccm"] * conversion / volume(fill_h), True]

    rt = temperature.loc[temperature["lab"].eq(lab)].copy()
    rt["minute"] = np.floor(rt["hours"] * 60 + 1e-10).astype(int)
    tm = rt.groupby("minute")["T"].median()
    available = (tm.index.to_numpy() + 1) / 60
    assert available[0] <= X0.at[lab, "hours"] and available[-1] >= last_hour

    ini = np.array([X0.at[lab, "X_g_L"], chem0.at[lab, "YAN for plot"] / 1000,
                    chem0.at[lab, "GLUCOSE-320"], chem0.at[lab, "Fructose (derived)"],
                    10.010194, 0., 0.])
    assert np.isfinite(ini).all() and (ini >= 0).all()
    P0 = np.diag([max(.03, .2 * ini[0])**2, .015**2, 12.5, 37.5, 4., 0., 0.])
    P0[2, 3] = P0[3, 2] = -12.5
    start = X0.at[lab, "hours"]
    pulse_h = round(nutrition_h[lab], 10)
    assert start < pulse_h < last_hour
    grid = np.unique(np.round(np.r_[np.arange(start, last_hour, 1 / 12), observed.index, pulse_h], 10))
    pulse_index = np.searchsorted(grid, pulse_h)
    assert grid[pulse_index] == pulse_h
    volume_at_pulse = volume(pulse_h)
    doses = np.zeros(len(grid))
    doses[pulse_index] = pulse_n_g / volume_at_pulse
    indices = np.searchsorted(grid, observed.index)
    np.testing.assert_array_equal(grid[indices], observed.index)
    settings = dict(grid=grid, doses=doses, initial=ini, P0=P0, available=available,
                    temperature=tm.to_numpy(float), S0=ini[2] + ini[3])
    cases = {}
    for name, sensor in (("Interpolado", augmented), ("Sin rellenar", observed)):
        usable = sensor["valid"] & sensor["measured"].ge(.05)
        assimilate = usable & usable.shift(fill_value=False)
        y = np.full(len(grid), np.nan)
        valid, update = np.zeros(len(grid), bool), np.zeros(len(grid), bool)
        y[indices], valid[indices], update[indices] = sensor["measured"], sensor["valid"], assimilate
        assert not update[indices[0]]
        cases[name] = dict(settings, y=y, valid=valid, update=update)
    added = grid[cases["Interpolado"]["update"] & ~cases["Sin rellenar"]["update"]]
    assert set(added) <= {fill_h, fill_h + .5}
    audit = {"LAB": lab, "X inicial [g/L]": ini[0], "YAN inicial [mg/L]": ini[1] * 1000,
             "G inicial [g/L]": ini[2], "F inicial [g/L]": ini[3],
             "Volumen a 22 h [L]": volume(fill_h), "Horizonte [h]": last_hour,
             "CO₂ interpolado [g/(L h)]": augmented.at[fill_h, "measured"],
             "Correcciones con CSV": int(cases["Interpolado"]["update"].sum()),
             "Correcciones sin rellenar": int(cases["Sin rellenar"]["update"].sum()),
             "Horas de corrección añadidas": ", ".join(f"{hour:g}" for hour in added) or "—"}
    return cases, audit

data = {}
audit_rows = []
for lab in labs:
    data[lab], audit = prepare_lab(lab)
    audit_rows.append(audit)
audit_table = pd.DataFrame(audit_rows).set_index("LAB")
display(audit_table.round(4))
pulse_table = pd.DataFrame({
    "Hora local": nutrition,
    "Hora desde inoculación [h]": nutrition_h,
    "Volumen [L]": {lab: pulse_n_g / data[lab]["Interpolado"]["doses"].sum() for lab in labs},
    "Pulso nominal [mg N/L]": {lab: 1000 * data[lab]["Interpolado"]["doses"].sum() for lab in labs},
})
display(pulse_table.round(3))

## Captación de YAN, crecimiento y actualización

In [ ]:
# Reuse the equations from the vectorized G/F EKF. Supply one set of three
# rate coefficients per trajectory, with the other coefficients shared.
source05 = EKF / "05_zenteno_co2_variable_solubility.source.ipynb"
nb05 = nbformat.read(source05, as_version=4)
functions = {}
for cell in nb05.cells:
    if cell.cell_type == "code":
        for node in ast.parse(cell.source).body:
            if isinstance(node, ast.FunctionDef):
                functions[node.name] = ast.get_source_segment(cell.source, node)
for name in ("co2_saturation", "gas_exchange"):
    exec(compile(functions[name], str(source05), "exec"))

rates_source = functions["rates_ekf"]
for original, replacement in (
    ("def rates_ekf(w,t,T,S0):", "def rates_ekf(w,t,T,S0,rate_coefficients):"),
    ("theta['mu0']", "rate_coefficients[:, 0]"),
    ("theta['betaG0']", "rate_coefficients[:, 1]"),
    ("theta['betaF0']", "rate_coefficients[:, 2]"),
):
    assert rates_source.count(original) == 1
    rates_source = rates_source.replace(original, replacement)
# Keep the growth rate active with either sugar and guard its substrate demands.
for original, replacement in (
    ("mu=np.where((g<=0)|(f<=0),0.,mu)", "mu=np.where((g>0)|(f>0),mu,0.)"),
    ("mu/theta['Yxg']", "np.where(g>0,mu/theta['Yxg'],0.)"),
    ("mu/theta['Yxf']", "np.where(f>0,mu/theta['Yxf'],0.)"),
):
    assert rates_source.count(original) == 1
    rates_source = rates_source.replace(original, replacement)
exec(compile(rates_source, "<tasas G/F con el azúcar disponible>", "exec"))
available_sugar_rates = rates_ekf

# Split the potential growth term into YAN uptake and effective growth.
for original, replacement in (
    ("def rates_ekf(w,t,T,S0,rate_coefficients):", "def rates_ekf(w,t,T,S0,rate_coefficients,growth_share=1.):"),
    ("mu=np.where((g>0)|(f>0),mu,0.)",
     "mu=np.where((g>0)|(f>0),mu,0.)\n uptake=mu*x/theta['Yxn'];mu=mu*growth_share"),
    ("-mu*x/theta['Yxn']", "-uptake"),
):
    assert rates_source.count(original) == 1
    rates_source = rates_source.replace(original, replacement)
exec(compile(rates_source, "<tasas G/F con reparto del YAN>", "exec"))
partitioned_rates = rates_ekf

points = np.array([
    [.5, .1, 10., 20., 30., .2, 0.],
    [.5, .1, 0., 20., 30., .2, 0.],
    [.5, .1, 10., 0., 30., .2, 0.],
    [.5, .1, 0., 0., 30., .2, 0.],
])
coefficients = np.broadcast_to(nominal, (len(points), len(nominal)))
baseline = available_sugar_rates(points, 0., 20., 150., coefficients)
unpartitioned = partitioned_rates(points, 0., 20., 150., coefficients)
smaller_growth = partitioned_rates(points, 0., 20., 150., coefficients, growth_share=.2)
np.testing.assert_array_equal(unpartitioned, baseline)
np.testing.assert_array_equal(smaller_growth[:, 1], baseline[:, 1])
np.testing.assert_array_equal(smaller_growth[[1, 2], [2, 3]], 0.)
np.testing.assert_allclose(smaller_growth[:, 0], .2 * baseline[:, 0])  # No death in these cases.
assert (smaller_growth[:3, 2:4].sum(axis=1) > baseline[:3, 2:4].sum(axis=1)).all()


def transition(points, rate_coefficients, start, end, data, rates):
    """Integrate each trajectory with its own constant rates over this interval."""
    w = points * SCALE
    n = max(1, int(np.ceil((end - start) * 60 - 1e-7)))
    dt = (end - start) / n
    available = data["available"]
    temperatures = data["temperature"]
    for j in range(n):
        t = start + j * dt
        times = np.array([t, t + dt / 2, t + dt])
        indices = np.searchsorted(available, times + 1e-12, side="right") - 1
        if indices[0] < 0:
            raise ValueError("No hay temperatura causal al comienzo del intervalo")
        T0, Tm, T1 = temperatures[indices]
        S0 = data["S0"]
        k1 = rates(w, t, T0, S0, rate_coefficients)
        k2 = rates(w + dt * k1 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k3 = rates(w + dt * k2 / 2, t + dt / 2, Tm, S0, rate_coefficients)
        k4 = rates(w + dt * k3, t + dt, T1, S0, rate_coefficients)
        w = np.maximum(w + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6, 0.)
    return w / SCALE


def sensitivities(z, start, end, data, rates, relative_step=1e-4):
    state_step = 1e-5
    delta = np.eye(len(z)) * state_step
    lower = z - delta * (z >= state_step)[:, None]
    denominator = state_step * (1 + (z >= state_step))
    points = np.vstack([z, z + delta, lower])
    coefficients = np.tile(nominal, (len(points), 1))
    parameter_step = relative_step * nominal
    parameter_delta = np.diag(parameter_step)
    points = np.vstack([points, np.tile(z, (6, 1))])
    coefficients = np.vstack([coefficients, nominal + parameter_delta, nominal - parameter_delta])

    predicted = transition(points, coefficients, start, end, data, rates)
    J = ((predicted[1:8] - predicted[8:15]) / denominator[:, None]).T
    G = ((predicted[15:18] - predicted[18:21]) / (2 * parameter_step[:, None])).T
    return predicted[0], J, G

In [ ]:
from functools import partial


def replay(data, partition_yan):
    growth_share = 1.
    rates = partitioned_rates if partition_yan else available_sugar_rates
    pulse_audit = []
    grid = data["grid"]
    z = data["initial"] / SCALE
    P = data["P0"] / scale_outer
    states = np.empty((len(grid), len(z)))
    covariance = np.empty((len(grid), len(z), len(z)))
    diag = np.full((len(grid), 7), np.nan)
    yan = np.zeros((len(grid), 2))
    growth_shares = np.ones(len(grid))
    for i, end in enumerate(grid):
        if i:
            start = grid[i - 1]
            before_N = z[1]
            z, J, G = sensitivities(z, start, end, data, rates)
            yan[i, 0] = (z[1] - before_N) * SCALE[1] * 1000
            scaled_G = G * sigma_theta
            Q = scaled_G @ scaled_G.T
            P = J @ P @ J.T + Q
            P = (P + P.T) / 2

        dose = data["doses"][i]
        if dose > 0:
            residual_sugar = z[2:4] @ SCALE[2:4]
            alpha = float(np.clip(residual_sugar / data["S0"], 0., 1.)) if partition_yan else 1.
            prior_yan = z[1] * SCALE[1]
            growth_share = (growth_share * prior_yan + alpha * dose) / (prior_yan + dose)
            pulse_audit.append({"Pulso [h]": end, "G+F al pulso [g/L]": residual_sugar,
                                "YAN previo [mg/L]": prior_yan * 1000, "Dosis [mg/L]": dose * 1000,
                                "Fracción del pulso": alpha, "Fracción efectiva": growth_share})
            if partition_yan:
                rates = partial(partitioned_rates, growth_share=growth_share)
            z[1] += dose / SCALE[1]
        growth_shares[i] = growth_share
        if end > 0 and abs(end / .5 - round(end / .5)) < 1e-8:
            predicted = z[6] / .5
            innovation = data["y"][i] - predicted
            S = 4 * P[6, 6] + R
            assert S > 0
            clips = 0
            if data["update"][i]:
                K = 2 * P[:, 6] / S
                updated = z + K * innovation
                clips = np.count_nonzero(updated[:6] < 0)
                B = np.eye(len(z))
                B[:, 6] -= 2 * K
                P = B @ P @ B.T + np.outer(K, K) * R
                P = (P + P.T) / 2
                yan[i, 1] = (max(updated[1], 0.) - z[1]) * SCALE[1] * 1000
                z = np.maximum(updated, 0)
            diag[i] = [predicted, innovation, S, innovation**2 / S, clips,
                       data["update"][i], data["valid"][i]]
            z[6] = 0.
            P[6, :] = P[:, 6] = 0.
        states[i] = z * SCALE
        covariance[i] = P * scale_outer
    assert np.isfinite(states).all() and np.isfinite(covariance).all()
    eigenvalues = np.linalg.eigvalsh(covariance / scale_outer)
    assert np.all(eigenvalues[:, 0] >= -1e-9 * np.maximum(1., eigenvalues[:, -1]))
    assert (yan[:, 0] <= 1e-9).all()
    np.testing.assert_allclose(1000 * states[:, 1],
                               1000 * data["initial"][1] + np.cumsum(yan.sum(axis=1) + 1000 * data["doses"]),
                               atol=1e-8)
    np.testing.assert_array_equal(diag[:, 5] == 1, data["update"])
    assert np.all((growth_shares >= 0) & (growth_shares <= 1 + 1e-12))
    return (states, covariance, diag), yan, growth_shares, pulse_audit

## Ejecución y métricas

Se reproducen la variante «Cambio» del 16 y el modelo con reparto del YAN, con las mismas ventanas de CO₂ y muestras. El RMSE de CO₂ usa el pronóstico previo a cada corrección. Se reporta para todo el intervalo evaluable y también después del último pulso. Para LAB022–024 se muestra además la biomasa estimada en la última muestra (118,33 h) y en el pico posterior al pulso.

In [ ]:
inputs_by_lab = {lab: {"Observado": DATA[lab]} for lab in LABS}
inputs_by_lab.update(data)
LABS = (*LABS, *labs)
ROLE.update(dict.fromkeys(labs, "Holdout LAB022–024"))
models = {"EKF 16": False, "Reparto YAN": True}
results, yan_changes, shares, pulses = {}, {}, {}, []
assert set(inputs_by_lab) == {f"LAB{number:03d}" for number in range(16, 25)}
for lab, cases in inputs_by_lab.items():
    for case, inputs in cases.items():
        for model, partition in models.items():
            key = lab, case, model
            results[key], yan_changes[key], shares[key], pulse_audit = replay(inputs, partition)
            if partition:
                pulses.extend({"LAB": lab, "Señal": case, **row} for row in pulse_audit)
        first_pulse = np.flatnonzero(inputs["doses"])[0]
        for reference, experiment in zip(results[lab, case, "EKF 16"], results[lab, case, "Reparto YAN"]):
            np.testing.assert_array_equal(reference[:first_pulse + 1], experiment[:first_pulse + 1])
        print(f"{lab} · {case} · ambos modelos completos", flush=True)
display(pd.DataFrame(pulses).set_index(["LAB", "Señal", "Pulso [h]"]).round(4))


In [ ]:
def state_series(states, variable):
    if variable == "GF":
        return states[..., 2] + states[..., 3]
    return states[..., core.STATES.index(variable)]


def state_at(hour, inputs, states, growth_shares, partition):
    grid = inputs["grid"]
    j = np.searchsorted(grid, hour + 1e-9, side="right") - 1
    assert 0 <= j and hour <= grid[-1]
    if abs(hour - grid[j]) <= 1e-9:
        return states[j]
    rates = (partial(partitioned_rates, growth_share=growth_shares[j])
             if partition else available_sugar_rates)
    predicted = transition((states[j] / SCALE)[None, :], nominal[None, :],
                           grid[j], hour, inputs, rates)
    return predicted[0] * SCALE


recent_samples = {
    "X": (biomass, "X_g_L"),
    "N": (chemistry, "YAN for plot"),
    "G": (chemistry, "GLUCOSE-320"),
    "F": (chemistry, "Fructose (derived)"),
    "GF": (chemistry, "GLU-FRU-320"),
}
scores, post_rows = [], []
for (lab, case, model), result in results.items():
    inputs = inputs_by_lab[lab][case]
    states, _, diagnostics = result
    grid = inputs["grid"]
    pulse = np.flatnonzero(inputs["doses"] > 0)[-1]
    context = dict(Grupo=ROLE[lab], LAB=lab, Señal=case, Modelo=model)
    if lab in DATA:
        historical, _, _ = core.diagnostics(DATA, CHEM, lab, result)
        mask = inputs["valid"] & np.isfinite(diagnostics[:, 0])
        historical = historical.loc[historical.variable.isin({"CO2", "X", "N", "G", "F", "GF"})]
        scores.extend(historical[["variable", "unit", "n", "RMSE"]].assign(**context).to_dict("records"))
    else:
        observed = inputs_by_lab[lab]["Sin rellenar"]
        mask = observed["valid"] & (grid > .5)
        error = diagnostics[mask, 0] - observed["y"][mask]
        scores.append(dict(context, variable="CO2", unit="g/(L h)", n=len(error),
                           RMSE=np.sqrt(np.mean(error**2))))
        for variable, (frame, col) in recent_samples.items():
            samples = frame.loc[frame.lab.eq(lab) & frame["sample"].ne("01")].dropna(subset=[col])
            predicted = np.array([state_at(t, inputs, states, shares[lab, case, model], models[model]) for t in samples.hours])
            factor = 1000 if variable == "N" else 1
            error = factor * state_series(predicted, variable) - samples[col].to_numpy()
            scores.append(dict(context, variable=variable, unit="mg/L" if variable == "N" else "g/L",
                               n=len(error), RMSE=np.sqrt(np.mean(error**2))))

    observed = inputs if lab in DATA else inputs_by_lab[lab]["Sin rellenar"]
    after = mask & (grid > grid[pulse])
    forecast_error = diagnostics[after, 0] - observed["y"][after]
    uptake, correction = yan_changes[lab, case, model][pulse + 1:].sum(axis=0)
    peak = pulse + np.argmax(states[pulse:, 0])
    row = {**context, "Último pulso [h]": grid[pulse],
           "X al pulso [g/L]": states[pulse, 0], "Pico X [g/L]": states[peak, 0],
           "Hora del pico [h]": grid[peak], "Captación de YAN [mg/L]": -uptake,
           "Corrección neta de YAN [mg/L]": correction, "YAN final [mg/L]": 1000 * states[-1, 1],
           "RMSE de CO₂ pospulso [g/(L h)]": np.sqrt(np.mean(forecast_error**2))}
    if lab in labs:
        sample = biomass.loc[biomass.lab.eq(lab)].iloc[-1]
        row["X observada a 118,33 h [g/L]"] = sample.X_g_L
        row["X estimada a 118,33 h [g/L]"] = state_at(
            sample.hours, inputs, states, shares[lab, case, model], models[model])[0]
    post_rows.append(row)

rmse = pd.DataFrame(scores)
assert np.isfinite(rmse.RMSE).all()
matched = rmse.groupby(["Grupo", "LAB", "Señal", "variable", "unit"])
assert matched.n.nunique().eq(1).all() and matched.size().eq(2).all()
comparison = rmse.pivot(index=["Grupo", "LAB", "Señal", "variable", "unit", "n"],
                        columns="Modelo", values="RMSE")[["EKF 16", "Reparto YAN"]]
comparison["Diferencia"] = comparison["Reparto YAN"] - comparison["EKF 16"]
post_pulse = pd.DataFrame(post_rows).set_index(["Grupo", "LAB", "Señal", "Modelo"])
with pd.option_context("display.max_rows", None):
    display(comparison.round(5))
    display(post_pulse.round(4))


## Trayectorias

Cada panel superpone el EKF 16 (gris) y la variante con reparto de YAN (azul). Se muestran el flujo de CO₂ antes de corregir y los estados después de corregir. Las filas corresponden a LAB019–021, LAB016–018 y LAB022–024. Las figuras de LAB022–024 muestran la corrida interpolada. Las tablas también incluyen la corrida con el hueco original.

In [ ]:
def legend_inside(ax, error_label):
    handles, labels = ax.get_legend_handles_labels()
    entries = dict(zip(labels, handles))
    assert len(labels) == len(entries)
    legend = ax.legend(entries.values(), entries.keys(), loc="best",
                       fontsize=8, framealpha=.85, title=error_label, title_fontsize=8)
    if error_label:
        legend.get_title().set_ha("left")


def draw_panel(variable, title, unit, symbol):
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    for ax, lab in zip(axes.flat, LABS):
        recent_lab = lab in labs
        case = "Interpolado" if recent_lab else "Observado"
        d = inputs_by_lab[lab][case]
        time = d["grid"]
        factor = 1000 if variable == "N" else 1
        for model, color, style in (("EKF 16", "#777777", "--"), ("Reparto YAN", "#0072B2", "-")):
            states, _, diagnostics = results[lab, case, model]
            if variable == "CO2":
                mask = np.isfinite(diagnostics[:, 0]) & (time > .5 if recent_lab else d["valid"])
                x, y = time[mask], diagnostics[mask, 0]
            else:
                x, y = time, factor * state_series(states, variable)
            label = "EKF 16" if model == "EKF 16" else symbol
            ax.plot(x, y, color=color, ls=style, lw=1.4, label=label)
        if variable == "CO2":
            if recent_lab:
                observed = inputs_by_lab[lab]["Sin rellenar"]
                windows = np.isfinite(diagnostics[:, 0])
                ax.plot(time[windows], observed["y"][windows], color="black", lw=1, label="Samples")
                synthetic = d["valid"] & ~observed["valid"]
                ax.scatter(time[synthetic], d["y"][synthetic], facecolors="none",
                           edgecolors="#D55E00", s=32, label="Interpolated")
            else:
                ax.plot(d["sensor"].t_h, d["sensor"].measured, color="black", lw=1, label="Samples")
        elif recent_lab and variable in recent_samples:
            frame, col = recent_samples[variable]
            samples = frame.loc[frame.lab.eq(lab)].dropna(subset=[col])
            ax.scatter(samples.hours, samples[col], color="black", s=23, label="Samples", zorder=5)
        elif not recent_lab and variable in core.CHEM_COLS:
            col = core.CHEM_COLS[variable]
            samples = CHEM.loc[CHEM.lab.eq(lab) & CHEM.t_h.gt(0) &
                               CHEM.t_h.le(d["horizon"])].dropna(subset=[col])
            if variable == "GF":
                samples = samples.loc[~samples.GF_zero_substitution.astype(str).str.lower().eq("true")]
            initial = state_series(d["initial"], variable)
            ax.scatter(np.r_[0., samples.t_h], factor * np.r_[initial, samples[col]],
                       color="black", s=23, label="Samples", zorder=5)
        bottom = 0.
        if variable == "GF" and not recent_lab:
            density = DENSITY.loc[DENSITY.lab.eq(lab) &
                                  DENSITY.t_h.between(0, d["horizon"])].dropna(subset=["density_g_l"])
            if len(density):
                from_density = 2.5 * (density.density_g_l - 1000)
                ax.scatter(density.t_h, from_density, marker="D", facecolors="none", edgecolors="#D55E00",
                           s=32, label="G+F from density", zorder=4)
                bottom = min(0., float(from_density.min()) * 1.15)
                ax.axhline(0, color="gray", lw=.6)
        for i, pulse in enumerate(time[d["doses"] > 0]):
            ax.axvline(pulse, color="gray", ls=":", lw=.8, label="N pulse" if i == 0 else "_nolegend_")
        score = rmse.loc[rmse.LAB.eq(lab) & rmse.Señal.eq(case) & rmse.Modelo.eq("Reparto YAN") &
                         rmse.variable.eq(variable), "RMSE"]
        error_label = f"RMSE (YAN split) = {score.iloc[0]:.3f} {unit}" if len(score) else None
        ax.set(title=lab, xlabel="Time [h]", ylabel=f"{title} [{unit}]", ylim=(bottom, None))
        ax.grid(alpha=.2)
        legend_inside(ax, error_label)
        assert sum(line.get_label() == symbol for line in ax.lines) == 1
    fig.suptitle(title, y=.99)
    fig.tight_layout(pad=.7, h_pad=1.2, w_pad=.9)
    plt.show()


for variable, title, unit, symbol in (
    ("CO2", "CO₂ flow", "g/(L h)", r"$\widehat{y}_{k|k-1}$"),
    ("X", "Biomass", "g/L", r"$\widehat{X}_{k|k}$"),
    ("N", "YAN", "mg/L", r"$\widehat{N}_{k|k}$"),
    ("G", "Glucose", "g/L", r"$\widehat{G}_{k|k}$"),
    ("F", "Fructose", "g/L", r"$\widehat{F}_{k|k}$"),
    ("GF", "Glucose + fructose", "g/L", r"$\widehat{G}_{k|k}+\widehat{F}_{k|k}$"),
):
    draw_panel(variable, title, unit, symbol)
